# WasteLens · Train the six-class Lab02 MLP

On Kaggle, enable **Internet** and choose **GPU T4 ×2** or **L4** under Accelerator. Training uses one GPU with CUDA mixed precision and stops if CUDA is unavailable. Use the PyTorch installation provided by Kaggle.

The Lab02 MLP keeps its RGB 64 × 64 input, hidden widths 128 and 32, ReLU and Xavier initialization. Its new head emits six logits for TrashNet. Validation macro-F1 selects the checkpoint before the test split is evaluated.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

KAGGLE_WORKING = Path("/kaggle/working")
ON_KAGGLE = KAGGLE_WORKING.is_dir()
KAGGLE_CHECKOUT = Path("/tmp/wastelens")
EPOCHS = 30
BATCH_SIZE = 64
DEVICE = "cuda" if ON_KAGGLE else "auto"

candidates = [Path.cwd(), *Path.cwd().parents]
if ON_KAGGLE:
    candidates.append(KAGGLE_CHECKOUT)
root = next((p for p in candidates if (p / "src/wastelens/train.py").is_file()), None)
if root is None and ON_KAGGLE:
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/Quang-Luu-2005/wastelens.git",
        str(KAGGLE_CHECKOUT),
    ], check=True)
    root = KAGGLE_CHECKOUT
if root is None:
    raise FileNotFoundError("Run from the WasteLens repo or import this notebook on Kaggle.")

sys.path.insert(0, str(root / "src"))
from wastelens.train import select_device, train

select_device(DEVICE)
subprocess.run([sys.executable, str(root / "scripts/prepare_trashnet.py")], cwd=root, check=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output_root = KAGGLE_WORKING / "wastelens-results" if ON_KAGGLE else root / "outputs"
output = output_root / f"lab02-mlp-{run_id}"

In [ ]:
import csv
import json
from collections import Counter

with (root / "data/splits/trashnet.csv").open(encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))
summary = json.loads((root / "data/splits/trashnet-summary.json").read_text(encoding="utf-8"))
image_root = root / "data/trashnet"
rows_by_split = {name: [row for row in rows if row["split"] == name] for name in ("train", "val", "test")}
class_names = [label for _, label in sorted({(int(row["class_id"]), row["label"]) for row in rows})]

assert len({row["path"] for row in rows}) == len(rows) == summary["retained_images"]
assert {name: len(items) for name, items in rows_by_split.items()} == summary["split_counts"]
assert all((image_root / row["path"]).is_file() for row in rows)

print(f"{'class':<12}{'train':>6}{'val':>6}{'test':>6}")
for label in class_names:
    counts = Counter(row["split"] for row in rows if row["label"] == label)
    print(f"{label:<12}{counts['train']:>6}{counts['val']:>6}{counts['test']:>6}")

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

thumb_width, thumb_height = 160, 120
sheet = Image.new("RGB", (3 * thumb_width, 2 * (thumb_height + 22)), "white")
draw = ImageDraw.Draw(sheet)
for index, label in enumerate(class_names):
    sample = next(row for row in rows_by_split["train"] if row["label"] == label)
    x = index % 3 * thumb_width
    y = index // 3 * (thumb_height + 22)
    with Image.open(image_root / sample["path"]) as image:
        thumbnail = image.convert("RGB").resize((thumb_width, thumb_height), Image.Resampling.LANCZOS)
        sheet.paste(thumbnail, (x, y))
    draw.text((x + 6, y + thumb_height + 4), label, fill="black")
display(sheet)

## Train and evaluate

Horizontal flips are applied on the training split. The held-out validation and test images use the same preprocessing as inference. The run exports a CPU-loadable checkpoint, class order, learning history, metrics, confusion matrix and per-image test predictions.

In [ ]:
metrics = train(
    image_root, root / "data/splits/trashnet.csv", output,
    device=DEVICE, epochs=EPOCHS, batch_size=BATCH_SIZE,
)
print(f"Best epoch: {metrics['best_epoch']} | Test images: {len(rows_by_split['test'])}")
for label, scores in metrics["test"]["per_class"].items():
    print(f"{label:10s} precision={scores['precision']:.3f}  recall={scores['recall']:.3f}  f1={scores['f1']:.3f}  n={scores['support']}")

## Review the training and test results

Rows in the confusion matrix are true classes and columns are predicted classes. Review the lowest-recall classes and their errors when writing the Lab03 report.

In [ ]:
import matplotlib.pyplot as plt

with (output / "history.csv").open(encoding="utf-8", newline="") as stream:
    history = list(csv.DictReader(stream))
epochs = [int(row["epoch"]) for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
for field, label in (("train_loss", "Train"), ("val_loss", "Validation")):
    axes[0].plot(epochs, [float(row[field]) for row in history], marker=".", label=label)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Training and validation loss")
axes[0].legend()
axes[1].plot(epochs, [float(row["val_macro_f1"]) for row in history], marker=".", color="#267b69")
axes[1].axvline(metrics["best_epoch"], color="#545454", linestyle="--", label="Selected epoch")
axes[1].set(xlabel="Epoch", ylabel="Macro-F1", ylim=(0, 1), title="Validation macro-F1")
axes[1].legend()
fig.savefig(output / "learning-curves.png", dpi=170)
plt.show()

matrix = metrics["test"]["confusion_matrix"]
fig, ax = plt.subplots(figsize=(6.4, 5.4), layout="constrained")
heatmap = ax.imshow(matrix, cmap="Blues")
for i, row in enumerate(matrix):
    for j, count in enumerate(row):
        ax.text(j, i, count, ha="center", va="center", color="white" if count > heatmap.norm.vmax / 2 else "#172438")
ax.set(xticks=range(len(class_names)), yticks=range(len(class_names)),
       xticklabels=class_names, yticklabels=class_names,
       xlabel="Predicted class", ylabel="Actual class", title="TrashNet test confusion matrix")
fig.colorbar(heatmap, ax=ax, label="Images")
fig.savefig(output / "confusion-matrix.png", dpi=170)
plt.show()

In [ ]:
with (output / "test-predictions.csv").open(encoding="utf-8", newline="") as stream:
    mistakes = [row for row in csv.DictReader(stream) if row["actual"] != row["predicted"]]
examples = sorted(mistakes, key=lambda row: float(row["confidence"]), reverse=True)[:6]
if examples:
    fig, axes = plt.subplots(2, 3, figsize=(10, 5.3), layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, examples):
        with Image.open(image_root / row["path"]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"{row['actual']} → {row['predicted']} ({float(row['confidence']):.0%})", fontsize=10)
    fig.suptitle("High-confidence test errors")
    fig.savefig(output / "test-errors.png", dpi=170)
    plt.show()
print(f"Run files: {output}")